In [2]:
!pip install requests haversine pandas numpy

In [3]:
import pandas as pd
import numpy as np
import math
import requests
from io import StringIO

# --- 1) Tải dữ liệu IBTrACS ---
csv_url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.ALL.list.v04r00.csv"

print("Đang tải dữ liệu IBTrACS... (mất vài chục giây)")
r = requests.get(csv_url, timeout=120)
r.raise_for_status()
text = r.text

# --- 2) Đọc dữ liệu vào pandas ---
df = pd.read_csv(StringIO(text), skiprows=[1], parse_dates=["ISO_TIME"], low_memory=False)

# --- 3) Lọc dữ liệu từ năm 2010–2024 ---
start = pd.Timestamp("2010-01-01")
end = pd.Timestamp("2024-12-31 23:59:59")
df = df[(df["ISO_TIME"] >= start) & (df["ISO_TIME"] <= end)]

# --- 4) Hàm tính khoảng cách Haversine (km) ---
def haversine(coord1, coord2):
    R = 6371  # Bán kính Trái đất (km)
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# --- 5) Tâm và bán kính vùng cần xét ---
province_centroid = (16.47, 107.6)  # lat, lon
R_km = 100  # bán kính ảnh hưởng (km)

# --- 6) Chuẩn bị dữ liệu ---
lat_col = "LAT"
lon_col = "LON"
id_col = "SID"

df = df[df[lat_col].notna() & df[lon_col].notna()].copy()
df[lon_col] = df[lon_col].apply(lambda x: x - 360 if x > 180 else x)

# --- 7) Tính toán ảnh hưởng từng cơn bão ---
candidate_storms = []

for storm_id, grp in df.groupby(id_col):
    lats = grp[lat_col].values
    lons = grp[lon_col].values

    distances = [haversine(province_centroid, (float(lat), float(lon))) for lat, lon in zip(lats, lons)]
    min_dist = np.nanmin(distances)

    if min_dist <= R_km:
        idx_min = np.argmin(distances)
        time_of_closest = grp.iloc[idx_min]["ISO_TIME"]

        affected_points = grp[np.array(distances) <= R_km].copy()
        if not affected_points.empty:
            affected_points["Ngay_mua"] = affected_points["ISO_TIME"].dt.floor("D")
            SoNgay_MuaLon = affected_points["Ngay_mua"].nunique()
        else:
            SoNgay_MuaLon = 1

        candidate_storms.append({
            "SID": storm_id,
            "NAME": grp.iloc[0].get("NAME", ""),
            "BASIN": grp.iloc[0].get("BASIN", ""),
            "min_dist_km": float(min_dist),
            "time_of_closest": time_of_closest,
            "SoNgay_MuaLon": SoNgay_MuaLon
        })

# --- 8) Tổng hợp kết quả ---
res_df = pd.DataFrame(candidate_storms)

if res_df.empty:
    print("Không có cơn bão nào trong phạm vi đã chọn.")
else:
    res_df["year_of_close"] = res_df["time_of_closest"].dt.year
    res_df["Ngày"] = res_df["time_of_closest"].dt.day
    res_df["Tháng"] = res_df["time_of_closest"].dt.month
    res_df["Năm"] = res_df["time_of_closest"].dt.year
    res_df["Loại_thiên_tai"] = "Bão"

    # Phân loại mức độ ảnh hưởng
    def classify_impact(dist):
        if dist <= 100:
            return "Rất mạnh (ảnh hưởng trực tiếp)"
        elif dist <= 250:
            return "Trung bình (ảnh hưởng gián tiếp)"
        elif dist <= 400:
            return "Yếu (ngoại biên)"
        else:
            return "Không ảnh hưởng"

    res_df["Mức_độ_tác_động"] = res_df["min_dist_km"].apply(classify_impact)

    # --- 9) Tổng hợp thống kê theo năm ---
    summary = (
        res_df.groupby("year_of_close")
        .agg(Số_bão_ảnh_hưởng=("SID", "count"),
             Tổng_số_ngày_mưa_lớn=("SoNgay_MuaLon", "sum"))
        .reset_index()
        .rename(columns={"year_of_close": "Năm"})
    )

    # --- 10) Xuất file kết quả ---
    res_df = res_df.sort_values("time_of_closest").reset_index(drop=True)
    res_df.to_csv("KetQua_Bao_IBTrACS_HaiDuong.csv", index=False, encoding="utf-8-sig")
    summary.to_csv("TongHop_MuaLon_TheoNam.csv", index=False, encoding="utf-8-sig")

    print(f"\nTổng số cơn bão ảnh hưởng (2010–2024) trong bán kính {R_km} km: {len(res_df)}")
    print("\n===== THỐNG KÊ THEO NĂM =====")
    print(summary.to_string(index=False))

    print("\n===== DANH SÁCH TOÀN BỘ CƠN BÃO ẢNH HƯỞNG =====")
    print(res_df.to_string(index=False))

Đang tải dữ liệu IBTrACS... (mất vài chục giây)

Tổng số cơn bão ảnh hưởng (2010–2024) trong bán kính 100 km: 15

===== THỐNG KÊ THEO NĂM =====
 Năm  Số_bão_ảnh_hưởng  Tổng_số_ngày_mưa_lớn
2011                 1                     2
2013                 1                     2
2015                 1                     1
2016                 2                     3
2017                 2                     2
2019                 1                     3
2020                 2                     3
2021                 2                     2
2022                 2                     2
2023                 1                     1

===== DANH SÁCH TOÀN BỘ CƠN BÃO ẢNH HƯỞNG =====
          SID      NAME BASIN  min_dist_km     time_of_closest  SoNgay_MuaLon  year_of_close  Ngày  Tháng  Năm Loại_thiên_tai                Mức_độ_tác_động
2011267N16111   HAITANG    WP    30.441167 2011-09-27 00:00:00              2           2011    27      9 2011            Bão Rất mạnh (ảnh hưởng trực tiếp